## A. EXTRACT

Baca ketiga sumber data menggunakan Spark DataFrame, yaitu:
1. Data transaksi dari file CSV sebanyak 5000 baris.
2. Data produk dari file JSON sebanyak 30 baris.
3. Data ulasan dari file CSV sebanyak 3500 baris.

Tampilkan jumlah baris masing-masing DataFrame dan struktur/schema dari ketiga data tersebut.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas6-ETL") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Membaca data transaksi
df_transaksi = spark.read.csv(
    "tugas6_transaksi.csv",
    header=True,
    inferSchema=True
)

# Membaca data produk
df_produk = spark.read.json(
    "tugas6_produk.json"
)

# Membaca data ulasan
df_ulasan = spark.read.csv(
    "tugas6_ulasan.csv",
    header=True,
    inferSchema=True
)

# Menampilkan jumlah baris
print("Jumlah transaksi :", df_transaksi.count())
print("Jumlah produk    :", df_produk.count())
print("Jumlah ulasan    :", df_ulasan.count())

# Menampilkan schema
print("\n=== Schema Transaksi ===")
df_transaksi.printSchema()

print("\n=== Schema Produk ===")
df_produk.printSchema()

print("\n=== Schema Ulasan ===")
df_ulasan.printSchema()

26/09/30 22:52:23 WARN Utils: Your hostname, asih2 resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/30 22:52:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
                                                                                

Jumlah transaksi : 5000
Jumlah produk    : 30
Jumlah ulasan    : 3500

=== Schema Transaksi ===
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)


=== Schema Produk ===
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)


=== Schema Ulasan ===
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



### Penjelasan

Pada tahap Extract, ketiga sumber data dibaca menggunakan Spark DataFrame. Data transaksi dan ulasan dibaca dari file CSV dengan header dan inferSchema, sedangkan data produk dibaca dari file JSON.

Hasil pembacaan menunjukkan bahwa data terdiri dari 5000 transaksi, 30 produk, dan 3500 ulasan. Schema ditampilkan untuk mengetahui nama kolom dan tipe data yang akan digunakan pada tahap transformasi.

## B. TRANSFORM – JOIN

Lakukan join terhadap ketiga DataFrame dengan ketentuan:
1. Data transaksi dan ulasan dihubungkan berdasarkan order_id.
2. Karena tidak semua transaksi memiliki ulasan, gunakan LEFT JOIN antara transaksi dan ulasan.
3. Data transaksi dan produk dihubungkan berdasarkan product_id.
4. Karena seluruh transaksi memiliki product_id yang valid, gunakan INNER JOIN antara hasil transaksi-ulasan dengan data produk.
5. Tambahkan kolom total_pendapatan yang dihitung dari unit_terjual dikalikan harga.
6. Tampilkan hasil join.

In [2]:
from pyspark.sql.functions import col

# LEFT JOIN transaksi dengan ulasan
df_join = df_transaksi.join(
    df_ulasan,
    on="order_id",
    how="left"
)

# INNER JOIN transaksi dengan produk
df_join = df_join.join(
    df_produk,
    on="product_id",
    how="inner"
)

# Menghitung total pendapatan
df_join = df_join.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

df_join.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

### Penjelasan

Join pertama menggunakan LEFT JOIN antara data transaksi dan ulasan berdasarkan order_id. LEFT JOIN digunakan karena tidak semua transaksi memiliki ulasan, sehingga transaksi tanpa ulasan tetap dipertahankan.

Selanjutnya digunakan INNER JOIN antara hasil sebelumnya dengan data produk berdasarkan product_id. INNER JOIN digunakan karena setiap transaksi memiliki product_id yang valid.

Setelah proses join selesai, ditambahkan kolom total_pendapatan yang diperoleh dari perkalian jumlah unit terjual dengan harga produk.

## C. DATA QUALITY

Lakukan pengecekan dan penanganan data ulasan dengan ketentuan:
1. Identifikasi transaksi yang tidak memiliki ulasan setelah proses LEFT JOIN.
2. Isi nilai rating yang kosong dengan nilai 0.
3. Pembuatan penanda ada_ulasan harus dilakukan sebelum pengisian nilai kosong.
4. Jika rating tersedia maka ada_ulasan bernilai True.
5. Jika rating tidak tersedia maka ada_ulasan bernilai False.
6. Tampilkan order_id, rating, dan ada_ulasan.

In [3]:
# Membuat kolom ada_ulasan SEBELUM na.fill()
df_transform = df_join.withColumn(
    "ada_ulasan",
    col("rating").isNotNull()
)

# Mengisi rating yang null dengan 0
df_transform = df_transform.na.fill(
    {"rating": 0}
)

df_transform.select(
    "order_id",
    "rating",
    "ada_ulasan"
).show(10)

+--------+------+----------+
|order_id|rating|ada_ulasan|
+--------+------+----------+
|     TX0|     1|      true|
|     TX1|     0|     false|
|     TX2|     2|      true|
|     TX3|     5|      true|
|     TX4|     0|     false|
|     TX5|     5|      true|
|     TX6|     4|      true|
|     TX7|     0|     false|
|     TX8|     5|      true|
|     TX9|     0|     false|
+--------+------+----------+
only showing top 10 rows



### Penjelasan

Kolom ada_ulasan dibuat terlebih dahulu berdasarkan kondisi rating. Jika rating memiliki nilai maka transaksi dianggap memiliki ulasan dan bernilai True, sedangkan rating yang kosong menghasilkan nilai False.

Setelah penanda ada_ulasan dibuat, nilai rating yang kosong diisi dengan 0. Nilai 0 digunakan untuk membedakan transaksi yang belum memiliki ulasan dari transaksi yang memiliki rating.

Urutan ini penting karena jika nilai rating diisi 0 terlebih dahulu, sistem tidak dapat lagi membedakan rating kosong dengan rating yang bernilai 0.

## D. LOAD

Simpan hasil akhir ETL ke HDFS dengan ketentuan:
1. Format penyimpanan adalah Parquet.
2. Data dipartisi berdasarkan kategori.
3. Lokasi penyimpanan adalah /user/asih2/tugas6/hasil_etl.
4. Verifikasi isi direktori HDFS.
5. Baca kembali data Parquet dari HDFS.
6. Tampilkan jumlah data setelah dibaca kembali dan schema-nya.

In [4]:
hdfs_path = "hdfs://localhost:9000/user/asih2/tugas6/hasil_etl"

# Simpan hasil ETL ke HDFS dalam format Parquet
# dan dipartisi berdasarkan kategori
df_transform.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(hdfs_path)

print("Data berhasil disimpan ke HDFS:", hdfs_path)

# Baca kembali data dari HDFS
df_hasil = spark.read.parquet(hdfs_path)

print("Jumlah data setelah dibaca kembali:", df_hasil.count())

df_hasil.printSchema()
df_hasil.show(10)

Data berhasil disimpan ke HDFS: hdfs://localhost:9000/user/asih2/tugas6/hasil_etl
Jumlah data setelah dibaca kembali: 5000
root
 |-- product_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- rating: integer (nullable = true)
 |-- harga: long (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- total_pendapatan: long (nullable = true)
 |-- ada_ulasan: boolean (nullable = true)
 |-- kategori: string (nullable = true)

+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|nama_produk|total_pendapatan|ada_ulasan|  kategori|
+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Produk-21|           5000

### Penjelasan

Data hasil transformasi disimpan ke HDFS menggunakan format Parquet dan dipartisi berdasarkan kategori. Penggunaan partisi membuat data tersusun berdasarkan masing-masing kategori.

Setelah proses penyimpanan selesai, data dibaca kembali dari HDFS untuk memastikan proses LOAD berhasil. Hasil pembacaan kembali menunjukkan jumlah data sebanyak 5000 baris, sehingga data berhasil disimpan dan dibaca kembali dengan baik.

### Penjelasan
hdfs dfs -ls -R /user/asih2/tugas6/hasil_etl 

(bigdata) asih2@asih2:~/praktikum-bigdata$ hdfs dfs -ls -R /user/asih2/tugas6/hasil_etl
-rw-r--r--   3 asih2 supergroup          0 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - asih2 supergroup          0 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 asih2 supergroup      16366 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/kategori=Elektronik/part-00000-0d1248e8-1855-49e2-a393-60e1e3518199.c000.snappy.parquet
drwxr-xr-x   - asih2 supergroup          0 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 asih2 supergroup      11133 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/kategori=Fashion/part-00000-0d1248e8-1855-49e2-a393-60e1e3518199.c000.snappy.parquet
drwxr-xr-x   - asih2 supergroup          0 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 asih2 supergroup      10392 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/kategori=Kesehatan/part-00000-0d1248e8-1855-49e2-a393-60e1e3518199.c000.snappy.parquet
drwxr-xr-x   - asih2 supergroup          0 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/kategori=Makanan
-rw-r--r--   3 asih2 supergroup       6859 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/kategori=Makanan/part-00000-0d1248e8-1855-49e2-a393-60e1e3518199.c000.snappy.parquet
drwxr-xr-x   - asih2 supergroup          0 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/kategori=Rumah Tangga
-rw-r--r--   3 asih2 supergroup       9269 2026-09-30 22:47 /user/asih2/tugas6/hasil_etl/kategori=Rumah Tangga/part-00000-0d1248e8-1855-49e2-a393-60e1e3518199.c000.snappy.parquet

Perintah hdfs dfs -ls -R digunakan untuk melihat isi direktori hasil ETL secara rekursif. Hasil verifikasi menunjukkan adanya file Parquet dan lima partisi kategori, yaitu Elektronik, Fashion, Kesehatan, Makanan, dan Rumah Tangga.

## E. INSIGHT

Analisis data hasil ETL untuk mengetahui kategori yang memiliki persentase transaksi dengan ulasan paling rendah.

Hitung:
1. Total transaksi pada setiap kategori.
2. Jumlah transaksi yang memiliki ulasan.
3. Persentase transaksi yang memiliki ulasan.
4. Tentukan kategori dengan persentase transaksi dengan ulasan paling rendah.
5. Berikan interpretasi bisnis sebanyak 2–3 kalimat.

In [5]:
from pyspark.sql.functions import col, count, sum as spark_sum, when

df_insight = df_hasil.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    spark_sum(
        when(col("ada_ulasan") == True, 1).otherwise(0)
    ).alias("transaksi_dengan_ulasan")
)

df_insight = df_insight.withColumn(
    "persentase_ulasan",
    col("transaksi_dengan_ulasan") /
    col("total_transaksi") * 100
)

df_insight.orderBy(
    col("persentase_ulasan").asc()
).show()

+------------+---------------+-----------------------+-----------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|     Makanan|            536|                    369|68.84328358208955|
|   Kesehatan|            961|                    662|68.88657648283039|
|     Fashion|           1035|                    726|70.14492753623188|
|Rumah Tangga|            815|                    575| 70.5521472392638|
|  Elektronik|           1653|                   1168|  70.659407138536|
+------------+---------------+-----------------------+-----------------+



### Hasil Analisis

Berdasarkan hasil analisis, kategori dengan persentase transaksi yang memiliki ulasan paling rendah adalah Makanan dengan persentase sebesar 68,84%.

Rinciannya:
- Makanan: 536 transaksi, 369 memiliki ulasan, 68,84%
- Kesehatan: 961 transaksi, 662 memiliki ulasan, 68,89%
- Fashion: 1035 transaksi, 726 memiliki ulasan, 70,14%
- Rumah Tangga: 815 transaksi, 575 memiliki ulasan, 70,55%
- Elektronik: 1653 transaksi, 1168 memiliki ulasan, 70,66%

### Interpretasi Bisnis

Kategori Makanan memiliki persentase transaksi dengan ulasan paling rendah, yaitu 68,84%, sehingga masih terdapat sekitar 31,16% transaksi yang belum memiliki ulasan. Kondisi ini dapat menjadi perhatian untuk meningkatkan jumlah ulasan pada kategori Makanan, misalnya melalui pengingat kepada pelanggan setelah transaksi selesai atau pemberian insentif untuk memberikan ulasan.

## Kesimpulan

Pada praktikum ini telah dilakukan proses ETL menggunakan PySpark yang meliputi Extract, Transform, Data Quality, Load, dan Insight. Data dari tiga sumber berhasil digabungkan menggunakan LEFT JOIN dan INNER JOIN, kemudian dilakukan perhitungan total pendapatan serta penanganan data ulasan yang kosong.

Hasil akhir berhasil disimpan ke HDFS dalam format Parquet dengan partisi berdasarkan kategori dan dapat dibaca kembali sebanyak 5000 baris. Berdasarkan hasil analisis, kategori Makanan memiliki persentase transaksi dengan ulasan paling rendah, yaitu sebesar 68,84%.